# College Football Upset Vulnerability — Stage 2: Team-Game Structure and Initial Pregame Features



## 0. Mount Google Drive and load Stage 1 data


In [ ]:
from google.colab import drive
from pathlib import Path
import json
import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/CFB_Upset_Project")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
INPUT_FILE = PROCESSED_DIR / "games_clean.csv"

if not INPUT_FILE.is_file():
    raise FileNotFoundError(
        f"Cannot find {INPUT_FILE}. Run Stage 1's export and copy the file "
        "into your Google Drive CFB_Upset_Project/data/processed/ folder."
    )

games = pd.read_csv(INPUT_FILE, parse_dates=["date"], low_memory=False)
print(f"Loaded {len(games):,} games with {len(games.columns)} columns")
print(f"Seasons: {games['season'].min()}–{games['season'].max()}")
print("Source:", INPUT_FILE)
display(games[["game_id", "season", "date", "away", "home", "score_away", "score_home"]].head())

Mounted at /content/drive
Loaded 18,300 games with 61 columns
Seasons: 2004–2025
Source: /content/drive/MyDrive/CFB_Upset_Project/data/processed/games_clean.csv


,game_id,season,date,away,home,score_away,score_home
0,1,2004,2004-08-28,Indiana State,Miami (OH),0,49
1,2,2004,2004-08-28,USC,Virginia Tech,24,13
2,3,2004,2004-09-02,UTEP,Arizona State,9,41
3,4,2004,2004-09-02,Buffalo,Eastern Michigan,34,37
4,5,2004,2004-09-02,Texas A&M,Utah,21,41


## 1. Validate the input and audit team appearances



In [ ]:
required = [
    "game_id", "season", "date", "game_type", "week", "home", "away",
    "neutral", "score_home", "score_away", "home_win", "home_margin"
]
missing_required = sorted(set(required) - set(games.columns))
if missing_required:
    raise ValueError(f"Missing required Stage 1 columns: {missing_required}")

if games["game_id"].duplicated().any():
    raise ValueError("game_id is not unique; investigate the Stage 1 export")
if games[required].drop(columns=["week"]).isna().any().any():
    raise ValueError("Missing required non-week values; check the Stage 1 export")
if not games["home_margin"].eq(games["score_home"] - games["score_away"]).all():
    raise ValueError("Home margin disagrees with final scores")
if not games["home_win"].eq((games["score_home"] > games["score_away"]).astype(int)).all():
    raise ValueError("home_win disagrees with final scores")

# Robustly normalize neutral from CSV without coercing arbitrary strings to True.
if not pd.api.types.is_bool_dtype(games["neutral"]):
    neutral_map = {"true": True, "false": False, "1": True, "0": False}
    games["neutral"] = games["neutral"].astype(str).str.strip().str.lower().map(neutral_map)
if games["neutral"].isna().any():
    raise ValueError("Unable to interpret some neutral-site values")
games["neutral"] = games["neutral"].astype(bool)

if len(games) != 18_300 or games["season"].min() != 2004 or games["season"].max() != 2025:
    print("NOTE: This export differs from the 18,300 games / 2004–2025 audited in Stage 1.")

all_teams = pd.concat([games["home"], games["away"]], ignore_index=True)
appearances = all_teams.value_counts().rename_axis("team").to_frame("appearances")
print("Unique team labels:", len(appearances))
print("Teams with fewer than 10 recorded games:", (appearances["appearances"] < 10).sum())
print("Neutral games:", games["neutral"].sum())
display(appearances.tail(12))

Unique team labels: 249
Teams with fewer than 10 recorded games: 26
Neutral games: 1022


,appearances
team,
Tarleton State,4
Utah Tech,3
Mississippi Valley State,3
Hofstra,3
Lindenwood,3
Yale,2
Lehigh,2
Northwestern Oklahoma State,1
San Diego,1


## 2. Convert each game to two team-game rows



In [ ]:
# Box-score columns available for both teams. These are postgame measures.
STAT_BASES = [
    "first_downs", "third_down_comp", "third_down_att",
    "fourth_down_comp", "fourth_down_att", "pass_comp", "pass_att",
    "pass_yards", "rush_att", "rush_yards", "total_yards",
    "fum", "int", "pen_num", "pen_yards", "possession"
]
usable_stats = [
    stat for stat in STAT_BASES
    if f"{stat}_home" in games.columns and f"{stat}_away" in games.columns
]
print(f"Using {len(usable_stats)} box-score statistic families:", usable_stats)


def to_team_games(games_df, stats):
    parts = []
    for side, other in [("home", "away"), ("away", "home")]:
        row = pd.DataFrame({
            "game_id": games_df["game_id"],
            "season": games_df["season"],
            "date": games_df["date"],
            "week": games_df["week"],
            "game_type": games_df["game_type"],
            "team": games_df[side],
            "opponent": games_df[other],
            "designated_home": (side == "home"),
            "neutral": games_df["neutral"],
            "venue": np.where(games_df["neutral"], "neutral", side),
            "points_for": games_df[f"score_{side}"],
            "points_against": games_df[f"score_{other}"],
            "margin": games_df[f"score_{side}"] - games_df[f"score_{other}"],
            "win": (games_df[f"score_{side}"] > games_df[f"score_{other}"]).astype(int),
        })
        # Preserve source metadata, but verify timestamps before using as pregame features.
        for base in ["rank", "conf"]:
            row[f"{base}_team"] = games_df.get(f"{base}_{side}")
            row[f"{base}_opponent"] = games_df.get(f"{base}_{other}")
        for stat in stats:
            row[f"{stat}_for"] = games_df[f"{stat}_{side}"]
            row[f"{stat}_against"] = games_df[f"{stat}_{other}"]
        parts.append(row)

    out = pd.concat(parts, ignore_index=True)
    out = out.sort_values(["team", "date", "game_id"], kind="stable").reset_index(drop=True)
    return out


team_games = to_team_games(games, usable_stats)
print(f"Team-game rows: {len(team_games):,} (expected {2 * len(games):,})")
print(f"Columns: {len(team_games.columns)}")
display(team_games[[
    "game_id", "season", "date", "team", "opponent", "venue",
    "points_for", "points_against", "margin", "win", "total_yards_for"
]].head(10))

Using 16 box-score statistic families: ['first_downs', 'third_down_comp', 'third_down_att', 'fourth_down_comp', 'fourth_down_att', 'pass_comp', 'pass_att', 'pass_yards', 'rush_att', 'rush_yards', 'total_yards', 'fum', 'int', 'pen_num', 'pen_yards', 'possession']
Team-game rows: 36,600 (expected 36,600)
Columns: 50


,game_id,season,date,team,opponent,venue,points_for,points_against,margin,win,total_yards_for
0,7605,2013,2013-10-26,Abilene Christian,New Mexico State,away,29,34,-5,0,574.0
1,7929,2014,2014-08-27,Abilene Christian,Georgia State,away,37,38,-1,0,498.0
2,8131,2014,2014-09-13,Abilene Christian,Troy,away,38,35,3,1,493.0
3,8798,2015,2015-09-03,Abilene Christian,Fresno State,away,13,34,-21,0,247.0
4,9710,2016,2016-09-03,Abilene Christian,Air Force,away,21,37,-16,0,360.0
5,10621,2017,2017-09-02,Abilene Christian,New Mexico,away,14,38,-24,0,282.0
6,10658,2017,2017-09-09,Abilene Christian,Colorado State,away,10,38,-28,0,333.0
7,11496,2018,2018-09-01,Abilene Christian,Baylor,away,27,55,-28,0,466.0
8,12373,2019,2019-08-31,Abilene Christian,North Texas,away,31,51,-20,0,456.0
9,13066,2019,2019-11-23,Abilene Christian,Mississippi State,away,7,45,-38,0,282.0


## 3. Test the two-way transformation



In [ ]:
counts = team_games.groupby("game_id").size()
assert len(team_games) == 2 * len(games)
assert counts.eq(2).all(), "Each game must produce exactly two rows"
assert team_games.groupby("game_id")["designated_home"].sum().eq(1).all()
assert team_games.groupby("game_id")["win"].sum().eq(1).all()
assert team_games.groupby("game_id")["margin"].sum().eq(0).all()
assert team_games.groupby("game_id")["points_for"].sum().eq(
    team_games.groupby("game_id")["points_against"].sum()
).all()
assert team_games.groupby("game_id")["neutral"].nunique().eq(1).all()

same_day_team = team_games.loc[
    team_games.duplicated(["team", "date"], keep=False),
    ["team", "date", "game_id", "opponent", "season"]
]
print("Same-team same-date records:", len(same_day_team))
if len(same_day_team):
    display(same_day_team.sort_values(["team", "date"]).head(20))
    raise ValueError(
        "Team appeared more than once on the same day. Review before lagged features: "
        "a chronological shift could leak another same-day result."
    )

print("All game/team consistency checks passed.")
# Familiar example (if available): one game shown from both perspectives.
example = team_games.loc[team_games["game_id"].eq(games.iloc[-1]["game_id"])]
display(example[["game_id", "team", "opponent", "venue", "points_for", "points_against", "margin", "win"]])

Same-team same-date records: 0
All game/team consistency checks passed.


,game_id,team,opponent,venue,points_for,points_against,margin,win
11753,18300,Indiana,Miami (FL),neutral,27,21,6,1
15899,18300,Miami (FL),Indiana,neutral,21,27,-6,0


## 4. Audit team-season history availability


In [ ]:
team_season_counts = (
    team_games.groupby(["season", "team"]).size()
    .rename("recorded_games").reset_index()
)
print("Team-season combinations:", len(team_season_counts))
print("Recorded games per team-season:")
display(team_season_counts["recorded_games"].describe().to_frame().T)
print("Team-seasons with fewer than 5 recorded games:",
      (team_season_counts["recorded_games"] < 5).sum())
display(team_season_counts.nsmallest(12, "recorded_games"))

Team-season combinations: 4444
Recorded games per team-season:


,count,mean,std,min,25%,50%,75%,max
recorded_games,4444.0,8.235824,5.513123,1.0,1.0,12.0,13.0,16.0


Team-seasons with fewer than 5 recorded games: 1682


,season,team,recorded_games
3,2004,Appalachian State,1
19,2004,Chattanooga,1
26,2004,Delaware,1
29,2004,Eastern Illinois,1
30,2004,Eastern Kentucky,1
32,2004,Eastern Washington,1
39,2004,Furman,1
41,2004,Georgia Southern,1
46,2004,Idaho State,1
48,2004,Illinois State,1


## 5. Create initial **pregame-only** features



In [ ]:
def add_initial_pregame_features(team_games_df):
    out = team_games_df.sort_values(["team", "season", "date", "game_id"], kind="stable").copy()
    groups = out.groupby(["team", "season"], sort=False)

    out["season_prior_games"] = groups.cumcount()
    # All win values are known for completed historical games.
    out["season_prior_wins"] = groups["win"].cumsum() - out["win"]
    out["season_prior_win_pct"] = (
        out["season_prior_wins"] / out["season_prior_games"].replace(0, np.nan)
    )

    # shift(1) must occur *within* each team and season; no current-game values.
    out["season_prior_avg_margin"] = groups["margin"].transform(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    )
    out["season_prior_avg_total_yards"] = groups["total_yards_for"].transform(
        lambda s: s.shift(1).expanding(min_periods=1).mean()
    )
    out["season_prior_yardage_observations"] = groups["total_yards_for"].transform(
        lambda s: s.notna().astype(int).cumsum() - s.notna().astype(int)
    )
    out["days_since_previous_season_game"] = groups["date"].diff().dt.days
    return out


team_games = add_initial_pregame_features(team_games)
preview_cols = [
    "season", "date", "team", "opponent", "win", "margin", "total_yards_for",
    "season_prior_games", "season_prior_wins", "season_prior_win_pct",
    "season_prior_avg_margin", "season_prior_avg_total_yards",
    "season_prior_yardage_observations", "days_since_previous_season_game"
]
example_team = "Kansas State" if "Kansas State" in set(team_games["team"]) else team_games["team"].iloc[0]
example_season = 2025
print(f"Pregame feature progression for {example_team}, season {example_season}:")
display(team_games.loc[
    team_games["team"].eq(example_team) & team_games["season"].eq(example_season),
    preview_cols
].head(8))

Pregame feature progression for Kansas State, season 2025:


,season,date,team,opponent,win,margin,total_yards_for,season_prior_games,season_prior_wins,season_prior_win_pct,season_prior_avg_margin,season_prior_avg_total_yards,season_prior_yardage_observations,days_since_previous_season_game
13001,2025,2025-08-23,Kansas State,Iowa State,0,-3,383.0,0,0,NaN,NaN,NaN,0,NaN
13002,2025,2025-08-30,Kansas State,North Dakota,1,3,461.0,1,0,0.000000,-3.000000,383.000000,1,7.0
13003,2025,2025-09-06,Kansas State,Army,0,-3,246.0,2,1,0.500000,0.000000,422.000000,2,7.0
13004,2025,2025-09-12,Kansas State,Arizona,0,-6,193.0,3,1,0.333333,-1.000000,363.333333,3,6.0
13005,2025,2025-09-27,Kansas State,UCF,1,14,434.0,4,1,0.250000,-2.250000,320.750000,4,15.0
13006,2025,2025-10-04,Kansas State,Baylor,0,-1,506.0,5,2,0.400000,1.000000,343.400000,5,7.0
13007,2025,2025-10-11,Kansas State,TCU,1,13,343.0,6,2,0.333333,0.666667,370.500000,6,7.0
13008,2025,2025-10-25,Kansas State,Kansas,1,25,371.0,7,3,0.428571,2.428571,366.571429,7,14.0


## 6. Verify that current-game results cannot leak into these features


In [ ]:
first_games = team_games["season_prior_games"].eq(0)
assert first_games.any()
assert team_games.loc[first_games, "season_prior_wins"].eq(0).all()
assert team_games.loc[first_games, [
    "season_prior_win_pct", "season_prior_avg_margin", "season_prior_avg_total_yards",
    "days_since_previous_season_game"
]].isna().all().all()
assert team_games.loc[first_games, "season_prior_yardage_observations"].eq(0).all()

# On randomly selected team-games, independently compare to strictly earlier games.
sample = team_games.loc[team_games["season_prior_games"] > 0].sample(
    n=min(40, (team_games["season_prior_games"] > 0).sum()), random_state=42
)
for row in sample.itertuples(index=False):
    history = team_games.loc[
        (team_games["team"] == row.team)
        & (team_games["season"] == row.season)
        & (team_games["date"] < row.date)
    ]
    assert len(history) == row.season_prior_games
    assert history["win"].sum() == row.season_prior_wins
    assert np.isclose(history["win"].mean(), row.season_prior_win_pct)
    assert np.isclose(history["margin"].mean(), row.season_prior_avg_margin)
    if history["total_yards_for"].notna().any():
        assert np.isclose(history["total_yards_for"].mean(), row.season_prior_avg_total_yards)
    else:
        assert pd.isna(row.season_prior_avg_total_yards)
    assert history["total_yards_for"].notna().sum() == row.season_prior_yardage_observations

print(f"Leakage sanity checks passed: {first_games.sum():,} first-of-season rows, "
      f"{len(sample)} independently verified later rows.")

Leakage sanity checks passed: 4,444 first-of-season rows, 40 independently verified later rows.


## 7. Return to **one row per game** for model training


In [ ]:
TEAM_FEATURES = [
    "season_prior_games", "season_prior_wins", "season_prior_win_pct",
    "season_prior_avg_margin", "season_prior_avg_total_yards",
    "season_prior_yardage_observations", "days_since_previous_season_game"
]


def select_side_features(team_games_df, side):
    flag = (side == "home")
    out = team_games_df.loc[
        team_games_df["designated_home"].eq(flag), ["game_id", "team", *TEAM_FEATURES]
    ].copy()
    out = out.rename(columns={"team": side, **{col: f"{side}_{col}" for col in TEAM_FEATURES}})
    return out


matchup_features = games[[
    "game_id", "season", "date", "week", "game_type", "home", "away",
    "neutral", "home_win", "home_margin"
]].copy()

matchup_features = matchup_features.merge(
    select_side_features(team_games, "home"), on=["game_id", "home"],
    how="left", validate="one_to_one"
).merge(
    select_side_features(team_games, "away"), on=["game_id", "away"],
    how="left", validate="one_to_one"
)

DIFFERENCE_BASES = [
    "season_prior_win_pct", "season_prior_avg_margin", "season_prior_avg_total_yards",
    "days_since_previous_season_game"
]
for metric in DIFFERENCE_BASES:
    matchup_features[f"diff_{metric}"] = (
        matchup_features[f"home_{metric}"] - matchup_features[f"away_{metric}"]
    )

# Explicitly define eligible columns, rather than passing an entire dataframe to a model.
INITIAL_MODEL_FEATURES = (
    ["neutral"]
    + [f"home_{c}" for c in TEAM_FEATURES]
    + [f"away_{c}" for c in TEAM_FEATURES]
    + [f"diff_{c}" for c in DIFFERENCE_BASES]
)

assert len(matchup_features) == len(games)
assert matchup_features["game_id"].is_unique
assert not matchup_features[["home", "away"]].isna().any().any()
assert not set(INITIAL_MODEL_FEATURES) & {"home_win", "home_margin", "points_for", "points_against", "score_home", "score_away"}

print(f"Matchups: {len(matchup_features):,}")
print(f"Initial candidate feature columns: {len(INITIAL_MODEL_FEATURES)}")
print("Missing pregame season-to-date values are expected in early-season games.")
display(matchup_features[[
    "season", "date", "away", "home", "neutral",
    "home_season_prior_games", "away_season_prior_games",
    "diff_season_prior_win_pct", "diff_season_prior_avg_margin", "home_win"
]].head(12))

Matchups: 18,300
Initial candidate feature columns: 19
Missing pregame season-to-date values are expected in early-season games.


,season,date,away,home,neutral,home_season_prior_games,away_season_prior_games,diff_season_prior_win_pct,diff_season_prior_avg_margin,home_win
0,2004,2004-08-28,Indiana State,Miami (OH),False,0,0,NaN,NaN,1
1,2004,2004-08-28,USC,Virginia Tech,False,0,0,NaN,NaN,0
2,2004,2004-09-02,UTEP,Arizona State,False,0,0,NaN,NaN,1
3,2004,2004-09-02,Buffalo,Eastern Michigan,False,0,0,NaN,NaN,1
4,2004,2004-09-02,Texas A&M,Utah,False,0,0,NaN,NaN,1
5,2004,2004-09-02,Boston College,Ball State,False,0,0,NaN,NaN,0
6,2004,2004-09-02,Tennessee-Martin,Western Michigan,False,0,0,NaN,NaN,1
7,2004,2004-09-02,Northwestern,TCU,False,0,0,NaN,NaN,1
8,2004,2004-09-03,Washington State,New Mexico,False,0,0,NaN,NaN,0
9,2004,2004-09-04,Northern Arizona,Arizona,False,0,0,NaN,NaN,1


## 8. Save outputs to Google Drive



In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
team_games_file = PROCESSED_DIR / "team_games.csv"
matchup_file = PROCESSED_DIR / "matchup_features_initial.csv"
features_file = PROCESSED_DIR / "initial_feature_columns.json"

team_games.to_csv(team_games_file, index=False)
matchup_features.to_csv(matchup_file, index=False)
with features_file.open("w", encoding="utf-8") as f:
    json.dump({
        "feature_columns": INITIAL_MODEL_FEATURES,
        "targets": ["home_win", "home_margin"],
        "notes": "First-pass season-to-date pregame features; missing first-game values intentional."
    }, f, indent=2)

print("Saved:")
for path in [team_games_file, matchup_file, features_file]:
    print(" -", path)
print(f"Team-games: {len(team_games):,}; matchups: {len(matchup_features):,}")

Saved:
 - /content/drive/MyDrive/CFB_Upset_Project/data/processed/team_games.csv
 - /content/drive/MyDrive/CFB_Upset_Project/data/processed/matchup_features_initial.csv
 - /content/drive/MyDrive/CFB_Upset_Project/data/processed/initial_feature_columns.json
Team-games: 36,600; matchups: 18,300
